# Embedding baseline with human-labeled examples

Each tagged transcript is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`. Each level is represented by the mean embedding of its labeled examples, and every transcript gets the level of the nearest mean.

**Data**: a csv with one row per transcript, its tagged text (`Transcript`), the level given by humans (`ability_label_name`) and, optionally, the reference level instructed to the LLM (`REFERENCE_COLUMN`), all Low / Medium / High.

**Protocol** (identical in `human_fgw.ipynb` and `human_embeddings.ipynb`):
* stratified $k$-fold cross-validation (stratified on the human labels);
* for each fold, $m$ transcripts per level are drawn from the other folds (all of them with `M = None`) as examples, labeled either by the **humans** or by the **reference** level; the transcripts of the fold are classified with them;
* each classification is scored against **both** the human and the reference level: accuracy, macro-F1, quadratic-weighted Cohen's $\kappa$ with a 95% bootstrap CI over transcripts;
* the folds and examples are saved in `splits/` and reloaded, so both methods use exactly the same ones;
* robustness: the whole cross-validation is repeated with `N_REPEATS` different random splits (mean ± SD);
* **unsupervised**: $k$-means with $K = 3$ on all the transcripts, without any label; each cluster is mapped to a level with the Hungarian algorithm (the one-to-one assignment that maximizes agreement with the label evaluated against, so the mapping can differ between the human and the reference level). ARI and AMI do not depend on that mapping. Repeated with `N_REPEATS` initializations.

In [ ]:
import os
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, kappa_with_ci, load_labeled_transcripts, load_or_create_examples,
                        load_or_create_split, map_clusters, plot_confusion)
from sklearn.cluster import KMeans

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
DIMENSION = "ability"           # names the splits and outputs

SPLIT_DIR = "splits"            # shared with the other notebook
OUTPUT_DIR = "results"

M = 10                          # labeled examples per level (None: all the transcripts of the other folds)
N_FOLDS = 5
SEED = 0                        # split of the main results
N_REPEATS = 5                   # random splits for the robustness check (seeds SEED, SEED + 1, ...)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean", as in embedding_baseline.ipynb
N_KMEANS_INIT = 10              # k-means restarts (unsupervised)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Labeled transcripts

In [ ]:
if REFERENCE_COLUMN is None:
    columns = pd.read_csv(LABELS_FILE, nrows=0).columns
    REFERENCE_COLUMN = next((c for c in columns if c not in (TEXT_COLUMN, LABEL_COLUMN)
                             and re.search("ref|instruct", c, re.IGNORECASE)), None)
print("reference column:", REFERENCE_COLUMN)

df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN)
LABELS = {"human": "level", **({"reference": "reference"} if REFERENCE_COLUMN else {})}  # label -> column of df
print(f"{len(df)} transcripts")
if M is not None:
    print(f"m = {M} examples per level from {N_FOLDS - 1} folds out of {N_FOLDS}: "
          f"about {int(np.ceil(M * N_FOLDS / (N_FOLDS - 1)))} transcripts per level are needed")
pd.DataFrame({label: df[column].value_counts() for label, column in LABELS.items()}).reindex(LEVELS).fillna(0).astype(int)

### Agreement between the human and the reference levels

How often humans recognize the instructed level: the context for the scores below.

In [ ]:
if "reference" in LABELS:
    agreement = kappa_with_ci(pd.DataFrame({"true": df["reference"], "pred": df["level"]}))
    print(f"exact agreement: {(df['reference'] == df['level']).mean():.3f}, "
          f"kappa = {agreement['kappa']:.2f} [{agreement['kappa_ci_low']:.2f}, {agreement['kappa_ci_high']:.2f}]")
    display(pd.crosstab(df["reference"], df["level"], rownames=["reference"], colnames=["human"])
            .reindex(index=LEVELS, columns=LEVELS, fill_value=0))

## Embeddings

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
X = model.encode(df["text"].tolist(), batch_size=64, show_progress_bar=True)
if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
vectors = dict(zip(df["id"], X))

## Nearest mean embedding

In [ ]:
def classify(folds, examples):
    # Predicted level of every transcript, from the examples of its own fold.
    predictions = []
    for fold in sorted(folds["fold"].unique()):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        centroids = np.array([np.mean([vectors[i] for i in support.loc[support["level"] == level, "id"]], axis=0)
                              for level in LEVELS])
        X_test = np.array([vectors[i] for i in test["id"]])
        distances = np.linalg.norm(X_test[:, None, :] - centroids[None, :, :], axis=2)
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
    return pd.concat(predictions, ignore_index=True)


def cluster(seed):
    # k-means (K = 3) on the embeddings of all the transcripts.
    labels = KMeans(n_clusters=len(LEVELS), n_init=N_KMEANS_INIT, random_state=seed).fit_predict(X)
    return pd.DataFrame({"id": df["id"], "cluster": labels})

## Classification (main split)

Examples labeled by humans or by the reference level, on the same folds; every classification is scored against both labels.

In [ ]:
METHOD = "embeddings"


def run(seed):
    # Predictions with examples labeled by each label, on the folds of this seed.
    folds, human_examples = load_or_create_split(df, DIMENSION, f"human_seed{seed}", SPLIT_DIR, M, N_FOLDS, seed)
    predictions = {}
    for source in LABELS:
        examples = human_examples if source == "human" else load_or_create_examples(
            folds, df.set_index("id").loc[folds["id"], LABELS[source]], f"{DIMENSION}_{source}_seed{seed}",
            SPLIT_DIR, M, seed)
        pred = classify(folds, examples).drop(columns="true")
        predictions[source] = pred.merge(df[["id", *LABELS.values()]], on="id")
    return predictions


def score_all(predictions):
    # Scores of every classification against every label.
    results = []
    for source, pred in predictions.items():
        for target, column in LABELS.items():
            results.append(evaluate({(DIMENSION, f"examples: {source}"): pred.assign(true=pred[column])},
                                    method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


predictions = run(SEED)
for source, pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_predictions_{DIMENSION}_{source}_examples.csv"), index=False)
results = score_all(predictions)
results.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_results_{DIMENSION}.csv"), index=False)

# mean and SD over folds
(results[results["fold"] != "all"].groupby(["examples", "evaluated against"])[["accuracy", "macro_f1", "kappa"]]
 .agg(["mean", "std"]).round(3))

All folds pooled, with the 95% bootstrap CI of $\kappa$:

In [ ]:
pooled = results[results["fold"] == "all"].copy()
pooled["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                            zip(pooled["kappa"], pooled["kappa_ci_low"], pooled["kappa_ci_high"])]
pooled.pivot(index="examples", columns="evaluated against", values=["accuracy", "macro_f1", "kappa [95% CI]"])

Confusion matrices (rows: label evaluated against; columns: examples used).

In [ ]:
plot_confusion({(f"vs {target}", f"examples: {source}"): pred.assign(true=pred[column])
                for target, column in LABELS.items() for source, pred in predictions.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_embedding_confusion_{DIMENSION}.png"))
plt.show()

## Robustness over random splits

The same cross-validation with `N_REPEATS` different splits (the folds and examples of each are saved too).

In [ ]:
repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    seed_results = score_all(predictions if seed == SEED else run(seed))
    repeats.append(seed_results[seed_results["fold"] == "all"].assign(seed=seed))
repeats = pd.concat(repeats, ignore_index=True)[["seed", "method", "examples", "evaluated against",
                                                 "accuracy", "macro_f1", "kappa"]]
repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_repeats_{DIMENSION}.csv"), index=False)
repeats.groupby(["examples", "evaluated against"])[["accuracy", "macro_f1", "kappa"]].agg(["mean", "std"]).round(3)

## Unsupervised: $k$-means and Hungarian mapping

No label is used to form the clusters; the labels only map clusters to levels for scoring.

In [ ]:
UNSUPERVISED = "none (k-means)"


def run_unsupervised(seed):
    return cluster(seed).merge(df[["id", *LABELS.values()]], on="id")


def score_unsupervised(pred):
    # Scores against every label, each with its own cluster-to-level mapping.
    results = []
    for target, column in LABELS.items():
        mapped = pred.assign(true=pred[column], pred=map_clusters(pred["cluster"], pred[column]))
        results.append(evaluate({(DIMENSION, UNSUPERVISED): mapped}, method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


unsupervised = run_unsupervised(SEED)
unsupervised.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_predictions_{DIMENSION}.csv"), index=False)
unsupervised_results = score_unsupervised(unsupervised)
unsupervised_results.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_results_{DIMENSION}.csv"), index=False)
unsupervised_results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in zip(
    unsupervised_results["kappa"], unsupervised_results["kappa_ci_low"], unsupervised_results["kappa_ci_high"])]
unsupervised_results[["evaluated against", "accuracy", "macro_f1", "kappa [95% CI]", "ARI", "AMI"]].round(3)

In [ ]:
plot_confusion({(f"vs {target}", UNSUPERVISED):
                unsupervised.assign(true=unsupervised[column], pred=map_clusters(unsupervised["cluster"], unsupervised[column]))
                for target, column in LABELS.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_confusion_{DIMENSION}.png"))
plt.show()

Over `N_REPEATS` initializations of $k$-means:

In [ ]:
unsupervised_repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    seed_results = score_unsupervised(unsupervised if seed == SEED else run_unsupervised(seed))
    unsupervised_repeats.append(seed_results.assign(seed=seed))
unsupervised_repeats = pd.concat(unsupervised_repeats, ignore_index=True)[
    ["seed", "method", "examples", "evaluated against", "accuracy", "macro_f1", "kappa", "ARI", "AMI"]]
unsupervised_repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_repeats_{DIMENSION}.csv"), index=False)
unsupervised_repeats.groupby("evaluated against")[["accuracy", "macro_f1", "kappa", "ARI", "AMI"]].agg(["mean", "std"]).round(3)